# Demographic, Clinical, and Laboratory Measurements from HIV Patients on TLD at Machakos Level 5 Hospital, Kenya (2024) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

*Citation*: Njoroge, M., Okalebo, F., & Nyamu, D. 2026. Demographic, Clinical, and Laboratory Measurements from HIV Patients on TLD at Machakos Level 5 Hospital, Kenya (2024). Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.s85d-x5d7/fair2.json'

# Load the croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display basic dataset metadata (non-dict access: use fields)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets and their respective fields, using their `@id`s.

The Croissant schema organizes data into **RecordSets**, each with a list of **Fields**. Below, we enumerate the `@id` and `name` for the dataset's record sets and their available fields.

In [ ]:
# List all record sets
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"  @id: {rs.id}; name: {rs.name}; description: {rs.description}")
    fields = list(rs.fields)
    print("    Fields:")
    for field in fields:
        print(f"      @id: {field.id}; name: {field.name}; data type: {field.data_type}")
    print()


## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Each record set and its fields must be referenced by their `@id`.

**Note:** If your record set has a large number of records, you may wish to inspect only the first few rows.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
# Print their ids for selection
print("Record set IDs:", record_set_ids)

# For this dataset, we expect the main record set to contain the clinical data, 
# e.g., '@id' ending with '/recordset/0' or similar. We'll demonstrate code for all sets found.

dataframes = {}
for rs in dataset.metadata.record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"\nLoaded {len(df)} records for record set id: {rs.id} (name: {rs.name})")
    if len(df.columns) > 0:
        print('  Columns:', df.columns.tolist())
    else:
        print('  (No tabular columns found in this record set)')

## 4. Exploratory Data Analysis (EDA)
Now, let's demonstrate simple processing and transformation operations for a specific record set.

We'll select a record set containing clinical/laboratory tabular data (adjust the record set `@id` as needed), choose a numeric field, and apply basic statistics and normalization.

Typical numeric fields in this dataset may include `age`, `serum_creatinine`, or similar.

In [ ]:
# Select a tabular record set by id (adjust as needed)
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    if 'demographic' in rs.name.lower() or 'clinical' in rs.name.lower() or True:
        main_record_set_id = rs.id
        break

# Display available fields, to select numeric field
print(f"Fields in record set '{main_record_set_id}':")
df = dataframes[main_record_set_id]
print(df.columns.tolist())

# Pick likely numeric fields for demonstration (adjust as present in data)
possible_numeric_fields = [c for c in df.columns if c.lower() in ['age', 'serum_creatinine', 'cd4_count', 'viral_load', 'creatinine', 'duration_on_art'] or df[c].dtype in ['int64','float64']]
print('Possible numeric fields:', possible_numeric_fields)
if len(possible_numeric_fields) == 0:
    raise Exception('No numeric fields found for EDA demonstration.')
numeric_field = possible_numeric_fields[0]

# Apply filtering and normalization
threshold = df[numeric_field].mean()  # Use mean as threshold example
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"\nFiltered records with '{numeric_field}' > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical variable, if available
group_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
if group_candidates:
    group_field = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print('No categorical field found for grouping.')

## 5. Visualization
Visualize distributions and relationships in the data, such as histograms, boxplots, or barplots for categorical grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of '{numeric_field}' in {main_record_set_id}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If a categorical field is available, boxplot numeric field by group
if group_candidates:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
We have ingested, explored, and visualized the HIV TLD cohort dataset using `mlcroissant`. Key outputs:

- Dataset metadata and structure parsed using Croissant schema
- Record sets, field ids, and types listed
- Tabular data extracted and loaded as pandas DataFrames
- Basic data analysis performed (e.g., filtering, normalization, grouping)
- Visualizations illustrate core distributions and demographic splits

This notebook serves as a template for FAIR dataset exploration with the Croissant schema, ensuring all elements are referenced by their persistent `@id` values, supporting reproducibility, transparency, and future analytical workflows.
